# SPARQL — User Guide

`StarLayerGraph` extends rdflib's SPARQL engine for RDF 1.2 and to conform to SPARQL 1.2. SPARQL 1.2 introduces triple-term-aware functions, quoted-triple patterns in queries and updates, Turtle 1.2-based shorthand syntax, and direction-tagged literals.

See the [Graphs guide](02-graphs.ipynb) for the `TripleTerm`/`DirLangString`/reification used in the examples below.

## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. This guide assumes StarLayer has been pip installed.

Run cells from top to bottom — later sections reuse variables from earlier ones.

In [1]:
from starlayergraph import StarLayerGraph, Namespace

EX = Namespace("http://example.org/")

## 1. Query semantics and built-in functions

StarLayerGraph supports SPARQL 1.2 query execution:
- Query over reified quoted triples
- Turtle 1.2 quoted-triple syntax in queries (`<<( ... )>>`)
- Binds variables for terms inside triple terms (`?s ?p ?o`)

The examples below run against the example graph, created here.

In [2]:
g_parsed = StarLayerGraph()
g_parsed.bind("ex", EX)
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

    # language-direction literals: AcmeCorp's name in English and Arabic
    ex:AcmeCorp ex:name "Acme Corporation"@en--ltr .
    ex:AcmeCorp ex:name "شركة أكمي"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> ;
      ex:source ex:HRSystem ;
      ex:confidence "high" .

    # anonymous inline annotation block (asserted triple)
    ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} .

    # named reifier with annotations
    ex:alice ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} .

    # named reifier without annotation block 
    ex:alice ex:worksWith ex:SalesTeam ~ ex:stmt2 .
    ex:stmt2 ex:asReported ex:HRSystem .
    ex:stmt2 ex:reportedDate "1/1/2026" .

    # reification shorthand << s p o >> (no parens): an anonymous, unasserted
    # reifier of the same base triple ex:claim reifies above - a triple
    # can have more than one independent reification
    << ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:LinkedIn .

    # unasserted quoted triple term
    ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
''', format='turtle12')
print("parsed triples:", len(g_parsed))
print()
print(g_parsed.serialize(format="turtle12"))

parsed triples: 20

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:AcmeCorp ex:name "شركة أكمي"@ar--rtl, "Acme Corporation"@en--ltr .

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} ;
    ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} ;
    ex:worksWith ex:SalesTeam ~ ex:stmt2 {| ex:asReported ex:HRSystem ; ex:reportedDate "1/1/2026" |} .

ex:claim ex:confidence "high" ;
    ex:source ex:HRSystem ;
    rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

<< ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:LinkedIn .



In [3]:
# SPARQL query to bind all terms inside a quoted triple
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?s ?p ?o ?source WHERE {
  ?claim rdf:reifies <<( ?s ?p ?o )>> .
  ?claim ex:source ?source .
  FILTER(?p = ex:worksFor)
}
ORDER BY ?claim ?s ?o
""")

# ?claim can be an anonymous reifier (a real BNode) as well as a named one,
# so print via n3() (handles either) rather than qname() (URIRefs only).
for row in rows:
    print(
        row.claim.n3(g_parsed.namespace_manager),
        g_parsed.qname(row.s),
        g_parsed.qname(row.p),
        g_parsed.qname(row.o),
        "Source: ", g_parsed.qname(row.source),
    )

_:N59a1094b6ca549c881d8d6e13f4f0d34 ex:alice ex:worksFor ex:AcmeCorp Source:  ex:LinkedIn
ex:claim ex:alice ex:worksFor ex:AcmeCorp Source:  ex:HRSystem


In [4]:
# detect triple-term values using isTRIPLE
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?claim ?p WHERE {
  ?claim ?p ?statement .
  FILTER( isTRIPLE(?statement) )
}
ORDER BY ?claim
""")

# ?claim can be an anonymous reifier (a real BNode) as well as a named one,
# so print via n3() (handles either) rather than qname() (URIRefs only).
for row in rows:
    print(row.claim.n3(g_parsed.namespace_manager), g_parsed.qname(row.p))

_:N59a1094b6ca549c881d8d6e13f4f0d34 rdf:reifies
_:N7db37770759c44838576c3fb8a4f8466 rdf:reifies
ex:AuditSystem ex:reported
ex:claim rdf:reifies
ex:stmt1 rdf:reifies
ex:stmt2 rdf:reifies


### 1.a Additional SPARQL 1.2 functions

- `TRIPLE(s, p, o)` — a functional way to write a triple term `<<( s p o )>>`
- `SUBJECT()`, `PREDICATE()`, `OBJECT()` — extract the parts out of a triple term
- `LANGDIR()`, `hasLANGDIR()`, `STRLANGDIR()` — read, check, and build a literal's base direction
- `LANG()`, `hasLANG()` — SPARQL 1.1 functions, now also aware of direction-tagged literals

In [5]:
# 1.a.1 TRIPLE() and SUBJECT()/PREDICATE()/OBJECT()
print("1.a.1 TRIPLE()/SUBJECT()/PREDICATE()/OBJECT()")
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?s ?p ?o WHERE {
  ex:claim rdf:reifies ?t .
  FILTER(?t = TRIPLE(ex:alice, ex:worksFor, ex:AcmeCorp))
  BIND(SUBJECT(?t) AS ?s)
  BIND(PREDICATE(?t) AS ?p)
  BIND(OBJECT(?t) AS ?o)
}
""")
for row in rows:
    print(g_parsed.qname(row.s), g_parsed.qname(row.p), g_parsed.qname(row.o))

# 1.a.2 The <<( s p o )>> literal syntax works directly in BIND() too - equivalent
# to TRIPLE(s, p, o) above, not just a WHERE-clause pattern form.
print()
print("1.a.2 <<( s p o )>> in BIND()")
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?tt WHERE {
  BIND( <<( ex:alice ex:worksFor ex:AcmeCorp )>> AS ?tt )
}
""")
for row in rows:
    print(row.tt)

1.a.1 TRIPLE()/SUBJECT()/PREDICATE()/OBJECT()
ex:alice ex:worksFor ex:AcmeCorp

1.a.2 <<( s p o )>> in BIND()
<<( ex:alice ex:worksFor ex:AcmeCorp )>>


In [6]:
# 1.a.3 LANGDIR() / hasLANGDIR() / LANG() / hasLANG() over AcmeCorp's two names
print("1.a.3 LANGDIR()/hasLANGDIR()/LANG()/hasLANG()")
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?s ?lang ?dir ?hasDir WHERE {
  ?s ex:name ?lit .
  BIND(LANG(?lit) AS ?lang)
  BIND(LANGDIR(?lit) AS ?dir)
  BIND(hasLANGDIR(?lit) AS ?hasDir)
}
ORDER BY ?s ?lang
""")
for row in rows:
    print(g_parsed.qname(row.s), row.lang, row.dir, row.hasDir)

# 1.a.4 STRLANGDIR() constructs a direction-tagged literal directly from plain strings
print()
print("1.a.4 STRLANGDIR()")
rows = g_parsed.query('SELECT ?lit ' \
'WHERE { BIND(STRLANGDIR("hi", "en", "ltr") AS ?lit) }')
for row in rows:
    print(row.lit.n3())

1.a.3 LANGDIR()/hasLANGDIR()/LANG()/hasLANG()
ex:AcmeCorp ar rtl true
ex:AcmeCorp en ltr true

1.a.4 STRLANGDIR()
"hi"@en--ltr


### 1.b Turtle annotation shorthand inside SPARQL queries

The `{| ?pred ?val |}`, `~ ?r`, and `<< s p o >>` Turtle syntax used to parse and serialize graphs also works directly inside a SPARQL 1.2 `WHERE` clause. This allows querying using the same shorthand a graph is serialized with, without expanding to `rdf:reifies`/`<<( )>>`.

In [7]:
# 1.b.1 {| ?pred ?val |}: query an anonymous reifier's annotations inline (asserted triple)
print("1.b.1 {| ?pred ?val |}")
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  ex:alice ex:likes ex:ProductABC {| ?pred ?val |}
  FILTER(?pred != rdf:reifies)
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

# 1.b.2 ~ ?r: bind the reifier itself for a named reifier
print()
print("1.b.2 ~ ?r")
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
SELECT ?r WHERE {
  ex:alice ex:worksWith ex:SalesTeam ~ ?r
}
""")
for row in rows:
    print(g_parsed.qname(row.r))

# 1.b.3 << s p o >> ?pred ?val: reification shorthand, no assertion required
print()
print("1.b.3 << s p o >>")
rows = g_parsed.query("""
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?pred ?val WHERE {
  << ex:alice ex:likes ex:ProductABC >> ?pred ?val
 
}
ORDER BY ?pred
""")
for row in rows:
    print(g_parsed.qname(row.pred), row.val)

1.b.1 {| ?pred ?val |}
ex:since 2020
ex:source http://example.org/CRM

1.b.2 ~ ?r
ex:stmt2

1.b.3 << s p o >>
ex:since 2020
ex:source http://example.org/CRM
rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>>


### 1.c The `VERSION "1.2"` query prologue directive

SPARQL 1.2 introduces an optional version number, allowing a query to document which version of SPARQL it is using.
A leading `VERSION "1.2"` line (before any `PREFIX`/`SELECT`) declares that a query uses SPARQL 1.2 features. Declaring a version that doesn't match what the query actually uses (e.g. `VERSION "1.1"` on a query containing a triple term) emits a `SPARQL12ConformanceWarning`.

In [8]:
import warnings

# 1.c.1 VERSION "1.2" declared, and the query does use RDF 1.2 syntax - no warning.
# ?stmt can match an anonymous reifier (a real BNode) as well as a named one,
# so print via n3() (handles either) rather than qname() (URIRefs only).
print("1.c.1 VERSION \"1.2\" matches usage")
rows = g_parsed.query("""
VERSION "1.2"
PREFIX ex: <http://example.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
""")
print([row.stmt.n3(g_parsed.namespace_manager) for row in rows])

# 1.c.2 VERSION "1.1" declared, but the query text itself uses a triple term - warns.
print()
print("1.c.2 VERSION \"1.1\" mismatch warns")
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    rows = g_parsed.query("""
    VERSION "1.1"
    PREFIX ex: <http://example.org/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    SELECT ?stmt WHERE { ?stmt rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
    """)
    list(rows)
    for w in caught:
        print(f"{w.category.__name__}: {w.message}")

1.c.1 VERSION "1.2" matches usage
['ex:claim', '_:N59a1094b6ca549c881d8d6e13f4f0d34']

1.c.2 VERSION "1.1" mismatch warns


## 2. SPARQL 1.2 Update

`INSERT DATA`/`DELETE DATA` allow for the use of SPARQL 1.2 syntax, including triple-term patterns.

In [9]:
g_update = StarLayerGraph()
g_update.bind("ex", EX)

# INSERT DATA with a ground triple term - the base triple is not asserted.
g_update.update("""
    PREFIX ex: <http://example.org/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    INSERT DATA { ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> }
""")
print("triples:", len(g_update))
print()
print(g_update.serialize(format="turtle12"))

triples: 1

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



In [10]:
# INSERT ... WHERE: a triple-term pattern with variables, matched first,
# then instantiated into the INSERT template.
g_update.update("""
    PREFIX ex: <http://example.org/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    INSERT { ?stmt ex:confidence "high" }
    WHERE  { ?stmt rdf:reifies <<( ?s ex:worksFor ?o )>> }
""")
print(g_update.serialize(format="turtle12"))

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim ex:confidence "high" ;
    rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



## Further Reading

1. **[Getting Started](01-getting-started.ipynb)** — install, first parse, first query, first validate.
2. **[Graphs](02-graphs.ipynb)** — `TripleTerm`/`DirLangString` semantics, Turtle 1.2 reification syntax.
3. **SPARQL** — this guide.
   - 3.a **[SPARQL rules (pending)](03a-sparql-rules-pending.md)** — SPARQL-RL (SRL), a separate Datalog-style rules language, deliberately out of scope for this project.
   - 3.b **[SPARQL inferencing](03b-sparql-inferencing.ipynb)** — `.query(..., entailment="rdfs"|"owl-rl")`, a per-call choice between query-time rdfs:subClassOf rewrite and materialize-query-discard for full RDFS/OWL-RL.
5. **Other**
   - 5.c **[SPARQL queries as RDF](05c-sparql-query-as-rdf.ipynb)** — treating a query itself as RDF data you can encode, inspect, edit, and validate, a separate concern from writing or running one.